In [1]:
!nvidia-smi


Sat Feb  7 23:54:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   69C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


True
Tesla T4


In [3]:
!pip -q install "numpy==1.26.4"


In [4]:
!pip -q install -U "pandas==2.2.2" "scikit-learn==1.6" "datasets==2.20.0" "tqdm==4.67" "transformers==5.1.0" "accelerate>=1.1.0"


In [5]:
import torch, pandas, transformers, datasets, accelerate, sklearn
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0))
print("torch:", torch.__version__)
print("pandas:", pandas.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)
print("sklearn:", sklearn.__version__)


cuda: True Tesla T4
torch: 2.9.0+cu126
pandas: 2.2.2
transformers: 5.1.0
datasets: 2.20.0
accelerate: 1.12.0
sklearn: 1.6.0


In [6]:
from google.colab import files
files.upload()
#Upload train.csv, val.csv, test.csv

KeyboardInterrupt: 

In [7]:
import os
print(os.listdir("/content"))


['.config', 'test.csv', 'baseline_colab_model', 'train.csv', 'val.csv', 'sample_data']


In [8]:
from datasets import Dataset
import pandas as pd
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256

def load_split(path):
    df = pd.read_csv(path).dropna(subset=["content", "label"])
    df["label"] = df["label"].astype(int)
    return Dataset.from_pandas(df[["content", "label"]])

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = load_split("/content/train.csv")
val_ds = load_split("/content/val.csv")
test_ds = load_split("/content/test.csv")

def tok(batch):
    return tokenizer(
        batch["content"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

train_ds = train_ds.map(tok, batched=True, remove_columns=["content"])
val_ds = val_ds.map(tok, batched=True, remove_columns=["content"])
test_ds = test_ds.map(tok, batched=True, remove_columns=["content"])

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

len(train_ds), len(val_ds), len(test_ds)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/50809 [00:00<?, ? examples/s]

Map:   0%|          | 0/6351 [00:00<?, ? examples/s]

Map:   0%|          | 0/6352 [00:00<?, ? examples/s]

(50809, 6351, 6352)

In [9]:
import numpy as np
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

training_args = TrainingArguments(
    output_dir="baseline_colab_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.011456,0.024062,0.992600,0.993019,0.990599,0.991808


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=3176, training_loss=0.05395272109535539, metrics={'train_runtime': 1200.176, 'train_samples_per_second': 42.335, 'train_steps_per_second': 2.646, 'total_flos': 3365268029156352.0, 'train_loss': 0.05395272109535539, 'epoch': 1.0})

In [10]:
import os, json

val_metrics = trainer.evaluate()
test_metrics = trainer.evaluate(test_ds)

os.makedirs("baseline_colab_model", exist_ok=True)

model.save_pretrained("baseline_colab_model")
tokenizer.save_pretrained("baseline_colab_model")

with open("baseline_colab_model/val_metrics.json", "w") as f:
    json.dump(val_metrics, f, indent=2)

with open("baseline_colab_model/test_metrics.json", "w") as f:
    json.dump(test_metrics, f, indent=2)

val_metrics, test_metrics


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

({'eval_loss': 0.024062499403953552,
  'eval_accuracy': 0.9925995906156511,
  'eval_precision': 0.9930191972076788,
  'eval_recall': 0.9905988857938719,
  'eval_f1': 0.9918075649294056,
  'eval_runtime': 47.5102,
  'eval_samples_per_second': 133.676,
  'eval_steps_per_second': 8.356,
  'epoch': 1.0},
 {'eval_loss': 0.02549665793776512,
  'eval_accuracy': 0.9914987405541562,
  'eval_precision': 0.9906021580229725,
  'eval_recall': 0.9906021580229725,
  'eval_f1': 0.9906021580229725,
  'eval_runtime': 44.8913,
  'eval_samples_per_second': 141.497,
  'eval_steps_per_second': 8.844,
  'epoch': 1.0})

In [ ]:
!zip -r baseline_colab_model.zip baseline_colab_model
from google.colab import files
files.download("baseline_colab_model.zip")


  adding: baseline_colab_model/ (stored 0%)
  adding: baseline_colab_model/config.json (deflated 49%)
  adding: baseline_colab_model/val_metrics.json (deflated 44%)
  adding: baseline_colab_model/checkpoint-3176/ (stored 0%)
  adding: baseline_colab_model/checkpoint-3176/config.json (deflated 49%)
  adding: baseline_colab_model/checkpoint-3176/trainer_state.json (deflated 75%)
  adding: baseline_colab_model/checkpoint-3176/model.safetensors (deflated 8%)
  adding: baseline_colab_model/checkpoint-3176/scheduler.pt (deflated 62%)
  adding: baseline_colab_model/checkpoint-3176/training_args.bin (deflated 53%)
  adding: baseline_colab_model/checkpoint-3176/rng_state.pth (deflated 26%)
  adding: baseline_colab_model/checkpoint-3176/optimizer.pt